# AirIntel – Notebook 12: Deployment & Prediction Engine

**Objective**: Transform trained machine learning models into a modular, production-ready prediction pipeline. This notebook serves as the backend intelligence layer for AirIntel, supporting both **Scientific Mode** (full feature set with measured pollutants) and **Public Mode** (deployment-ready interface for future weather-driven forecasting without fabricating pollutant concentrations).

## Deployment Architecture & Flow

```text
User Input
    │
    ▼
Input Validation
    │
    ▼
Feature Builder (Spatial, Temporal & Weather Derivations)
    │
    ├── Scientific Mode (Full Pollutant Data)
    │         │
    │         ▼
    │   Prediction Models (LightGBM & CatBoost)
    │         │
    │         ▼
    │   Risk Engine & SHAP Explanation
    │         │
    │         ▼
    │   Recommendation Engine
    │
    └── Public Mode (Weather/Location Inputs)
              │
              ▼
        Deployment Payload Preparation (Deferred Forecast Response)
    │
    ▼
JSON Output / Dashboard Integration
```

## 01. Imports

Load core software development, machine learning, datetime, and serialization libraries.

In [1]:
# Import libraries
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import json
import math
import time
from datetime import datetime, timezone
import joblib
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm
import shap
try:
    import catboost
except ImportError:
    pass

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 02. Configuration

Define directory paths, deployment environment parameters, and random state seed.

In [2]:
# Configuration paths
PROJECT_DIR = Path(r"c:\Users\kayri\OneDrive - IIT BHU\Documents\Indian_Air_Quality_Project")
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
OPTIMIZED_DIR = MODEL_DIR / "optimized"
DEPLOYMENT_DIR = MODEL_DIR / "deployment"
REPORT_DIR = PROJECT_DIR / "reports"
TABLE_DIR = REPORT_DIR / "tables"

DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f"Deployment Directory: {DEPLOYMENT_DIR}")

Deployment Directory: c:\Users\kayri\OneDrive - IIT BHU\Documents\Indian_Air_Quality_Project\models\deployment


## 03. Load Deployment Assets

Verify target directories and environment initialization.

In [3]:
# Verify asset loading readiness
print(f"Model Directory exists: {MODEL_DIR.exists()}")
print(f"Optimized Directory exists: {OPTIMIZED_DIR.exists()}")
print(f"Data Directory exists: {DATA_DIR.exists()}")

Model Directory exists: True
Optimized Directory exists: True
Data Directory exists: True


## 04. Load Optimized Models

Inspect paths for optimized regression and classification models.

In [4]:
# Check model file paths
reg_path = OPTIMIZED_DIR / "lightgbm_regressor.pkl"
cls_path = OPTIMIZED_DIR / "catboost_classifier.pkl"

print(f"Regression model file exists: {reg_path.exists()}")
print(f"Classification model file exists: {cls_path.exists()}")

Regression model file exists: True
Classification model file exists: True


## 05. Load Encoders

Load label encoders for AQI classification categories.

In [5]:
# Load label encoder
le_path = MODEL_DIR / "label_encoder.pkl"
label_encoder = joblib.load(le_path)
print(f"Loaded target classes: {list(label_encoder.classes_)}")

Loaded target classes: ['Good', 'Hazardous', 'Moderate', 'Unhealthy', 'Unhealthy_Sensitive', 'Very_Unhealthy']


## 06. Load Metadata

Load feature schema, dataset reference medians, and city geographic coordinates.

In [6]:
# Load selected features and dataset reference medians
selected_features = joblib.load(OPTIMIZED_DIR / "selected_features.pkl")
df_sample = pd.read_parquet(DATA_DIR / "processed" / "airintel_ml_final.parquet")

# Compute numeric medians and categorical modes for optional feature fallback
feature_medians = {}
for col in selected_features:
    if col in df_sample.columns:
        if pd.api.types.is_numeric_dtype(df_sample[col]):
            feature_medians[col] = float(df_sample[col].median())
        else:
            feature_medians[col] = str(df_sample[col].mode()[0])

# City coordinates mapping
city_coords = df_sample.groupby('City')[['Latitude', 'Longitude']].mean().to_dict('index')

print(f"Loaded {len(selected_features)} selected features.")
print(f"City metadata mapped for {len(city_coords)} cities.")

Loaded 36 selected features.
City metadata mapped for 29 cities.


## 07. Deployment Configuration

Set runtime deployment parameters, operational prediction modes, and API version metadata.

In [7]:
# Global Deployment Settings
PREDICTION_ENGINE_NAME = "AirIntel v1.0"
MODEL_VERSION = "v1.0"
PIPELINE_VERSION = "v1.0"
DEPLOYMENT_VERSION = "1.0.0"
PREDICTION_MODES = ["scientific", "public"]

print(f"Engine: {PREDICTION_ENGINE_NAME} | Model Version: {MODEL_VERSION} | Pipeline: {PIPELINE_VERSION}")
print(f"Supported Modes: {PREDICTION_MODES}")

Engine: AirIntel v1.0 | Model Version: v1.0 | Pipeline: v1.0
Supported Modes: ['scientific', 'public']


## Public Mode Architecture

• Public users generally do not know or track real-time pollutant concentrations (PM2.5, PM10, NO2, SO2, CO, O3).
• Artificially imputing pollutant concentrations with statistical medians or synthetic values would reduce scientific validity.
• Therefore, Public Mode is intentionally designed as a deployment-ready interface that prepares validated inputs and feature engineering.
• Future versions of AirIntel will integrate a dedicated weather-based AQI forecasting model.
• The interface and feature processing pipeline are fully compatible with that future weather forecasting model.

---------------------------------
### SECTION A: MODEL LOADING
---------------------------------

## 08. Load Regression Model

Load the optimized LightGBM regression pipeline.

In [8]:
# Function to load regression pipeline
def load_regression_pipeline(filepath):
    if not Path(filepath).exists():
        raise FileNotFoundError(f"Regression model file not found at {filepath}")
    return joblib.load(filepath)

reg_pipeline = load_regression_pipeline(OPTIMIZED_DIR / "lightgbm_regressor.pkl")
print("Regression pipeline loaded successfully.")

Regression pipeline loaded successfully.


## 09. Load Classification Model

Load the optimized CatBoost classification pipeline.

In [9]:
# Function to load classification pipeline
def load_classification_pipeline(filepath):
    if not Path(filepath).exists():
        raise FileNotFoundError(f"Classification model file not found at {filepath}")
    return joblib.load(filepath)

cls_pipeline = load_classification_pipeline(OPTIMIZED_DIR / "catboost_classifier.pkl")
print("Classification pipeline loaded successfully.")

Classification pipeline loaded successfully.


## 10. Verify Model Integrity

Verify preprocessor and estimator steps for loaded pipeline objects.

In [10]:
# Model integrity verification function
def verify_pipeline_integrity(pipeline, expected_steps=['preprocessor', 'model']):
    steps = list(pipeline.named_steps.keys())
    is_valid = all(step in steps for step in expected_steps)
    return is_valid, steps

reg_valid, reg_steps = verify_pipeline_integrity(reg_pipeline)
cls_valid, cls_steps = verify_pipeline_integrity(cls_pipeline)

print(f"Regression Pipeline Valid: {reg_valid} (Steps: {reg_steps})")
print(f"Classification Pipeline Valid: {cls_valid} (Steps: {cls_steps})")

Regression Pipeline Valid: True (Steps: ['preprocessor', 'model'])
Classification Pipeline Valid: True (Steps: ['preprocessor', 'model'])


## 11. Display Deployment Summary

Summarize model deployment status and operational capabilities across modes.

In [11]:
# Display deployment summary table across modes
summary_data = [
    {"Mode": "Scientific", "Purpose": "Full analytical predictions using measured pollutants", "Prediction Available": "Yes", "Status": "Production Ready"},
    {"Mode": "Public", "Purpose": "Deployment-ready interface for future weather-based forecasting", "Prediction Available": "Deferred", "Status": "Architecture Ready"}
]
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
summary_df.to_csv(TABLE_DIR / "deployment_summary.csv", index=False)

      Mode                                                         Purpose Prediction Available             Status
Scientific           Full analytical predictions using measured pollutants                  Yes   Production Ready
    Public Deployment-ready interface for future weather-based forecasting             Deferred Architecture Ready


Why Model Loading Verification?

• Verifies model file presence and pipeline structure before serving requests.
• Prevents runtime exceptions during downstream inference.
• Ensures seamless transition between offline training and production serving.

---------------------------------
### SECTION B: INPUT VALIDATION
---------------------------------

## 12. Define Input Schema

Establish validation schemas for user input parameters.

In [12]:
# Define validation rules
VALID_CITIES = sorted(list(df_sample['City'].unique()))
VALID_SEASONS = ['Monsoon', 'Post_Monsoon', 'Winter', 'Summer']

NUMERIC_RANGES = {
    'Temp_2m_C': (-50.0, 60.0),
    'Humidity_Percent': (0.0, 100.0),
    'Surface_Pressure_hPa': (700.0, 1100.0),
    'Wind_Speed_10m_kmh': (0.0, 200.0),
    'Rain_mm': (0.0, 500.0),
    'Cloud_Cover_Percent': (0.0, 100.0),
    'Solar_Radiation_Wm2': (0.0, 1500.0)
}

print(f"Validation schema initialized for {len(VALID_CITIES)} cities and {len(NUMERIC_RANGES)} numeric weather fields.")

Validation schema initialized for 29 cities and 7 numeric weather fields.


## 13. Validate User Inputs

Construct validation logic for payload inspection across modes.

In [13]:
# Input payload validation function
def validate_user_payload(input_dict, mode="scientific"):
    errors = []
    cleaned_dict = input_dict.copy()

    # City check
    city = cleaned_dict.get('City')
    if not city or city not in VALID_CITIES:
        errors.append(f"Invalid or missing City '{city}'. Must be one of valid cities list.")

    # Weather numeric ranges check
    for param, (vmin, vmax) in NUMERIC_RANGES.items():
        if param in cleaned_dict:
            val = cleaned_dict[param]
            if not isinstance(val, (int, float)) or val < vmin or val > vmax:
                errors.append(f"Parameter '{param}' value {val} out of valid range [{vmin}, {vmax}].")

    is_valid = len(errors) == 0
    return is_valid, errors, cleaned_dict

# Test validation function
valid, errs, _ = validate_user_payload({'City': 'Delhi', 'Temp_2m_C': 25.0}, mode='public')
print(f"Payload validation status: {valid}, Errors: {errs}")

Payload validation status: True, Errors: []


## 14. Handle Missing Values

Implement fallback logic for optional weather parameters.

In [14]:
# Imputation handler for optional weather parameters
def apply_default_imputation(input_dict, feature_medians):
    imputed = input_dict.copy()
    for feature, default_val in feature_medians.items():
        if feature not in imputed or imputed[feature] is None:
            imputed[feature] = default_val
    return imputed

print("Default imputation engine ready for optional weather fields.")

Default imputation engine ready for optional weather fields.


## 15. Validate Feature Ranges

Test boundary check enforcement.

In [15]:
# Test out-of-bounds rejection
invalid_payload = {'City': 'Delhi', 'Temp_2m_C': 99.0}
is_valid, errors, _ = validate_user_payload(invalid_payload)
print(f"Invalid Payload Valid: {is_valid}")
print(f"Detected Errors: {errors}")

Invalid Payload Valid: False
Detected Errors: ["Parameter 'Temp_2m_C' value 99.0 out of valid range [-50.0, 60.0]."]


Why Input Validation?

• Rejects physically impossible or corrupt payloads before processing.
• Safeguards prediction reliability against out-of-range weather values.
• Reduces runtime failures in web applications and public APIs.

---------------------------------
### SECTION C: FEATURE PROCESSING
---------------------------------

## 16. Feature Preparation

Build feature derivation pipeline for cyclical, interaction, and spatial attributes.

In [16]:
# Feature engineering function
def derive_features(raw_dict, city_coords, feature_medians):
    d = raw_dict.copy()
    city = d.get('City', 'Delhi')

    # City geographical lookup
    if city in city_coords:
        d['Latitude'] = city_coords[city]['Latitude']
        d['Longitude'] = city_coords[city]['Longitude']
    else:
        d['Latitude'] = feature_medians.get('Latitude', 23.8)
        d['Longitude'] = feature_medians.get('Longitude', 80.9)

    d['Absolute_Latitude'] = abs(d['Latitude'])
    d['Lat_Long_Interaction'] = d['Latitude'] * d['Longitude']
    d['Northern_India'] = 1 if d['Latitude'] > 20.0 else 0

    # Temporal parameters
    month = d.get('Month', 6)
    d['Month_Sin'] = math.sin(2 * math.pi * month / 12)
    d['Month_Cos'] = math.cos(2 * math.pi * month / 12)

    hour = d.get('Hour', 12)
    d['Hour_Cos'] = math.cos(2 * math.pi * hour / 24)

    dow = d.get('Day_of_Week', 3)
    d['Weekday_Sin'] = math.sin(2 * math.pi * dow / 7)
    d['Weekday_Cos'] = math.cos(2 * math.pi * dow / 7)

    # Weather derived features
    temp = d.get('Temp_2m_C', feature_medians.get('Temp_2m_C', 25.0))
    humidity = d.get('Humidity_Percent', 50.0)
    d['Temp_Humidity'] = temp * humidity

    # Fill remaining missing selected features from medians
    for col, default_v in feature_medians.items():
        if col not in d or d[col] is None:
            d[col] = default_v

    return d

## 17. Feature Encoding

Ensure categorical features match pipeline expectations.

In [17]:
# Feature encoding handler
def encode_features_dataframe(df_row):
    df_out = df_row.copy()
    cat_cols = ['City', 'Season', 'Wind_Category', 'Latitude_Band', 'Longitude_Band', 'Time_of_Day', 'Humidity_Category']
    for col in cat_cols:
        if col in df_out.columns:
            df_out[col] = df_out[col].astype(str)
    return df_out

## 18. Feature Alignment

Align features into the exact column sequence expected by the preprocessor.

In [18]:
# Feature alignment function
def align_features(df_row, selected_features):
    return df_row[selected_features]

print("Feature alignment logic verified.")

Feature alignment logic verified.


## 19. Prediction Dataset Builder

Master feature processing function to prepare 1-row or multi-row input DataFrames.

In [19]:
# Master dataset builder function
def build_prediction_dataframe(input_dict, selected_features, city_coords, feature_medians, mode="scientific"):
    is_valid, errors, cleaned_dict = validate_user_payload(input_dict, mode=mode)
    if not is_valid:
        raise ValueError(f"Input validation failed: {errors}")

    derived_dict = derive_features(cleaned_dict, city_coords, feature_medians)
    df_single = pd.DataFrame([derived_dict])
    df_encoded = encode_features_dataframe(df_single)
    df_aligned = align_features(df_encoded, selected_features)
    return df_aligned

# Test dataset builder
test_payload = {'City': 'Delhi', 'Temp_2m_C': 32.5, 'Month': 11, 'Hour': 14}
df_proc = build_prediction_dataframe(test_payload, selected_features, city_coords, feature_medians, mode='scientific')
print(f"Processed DataFrame shape: {df_proc.shape}")

Processed DataFrame shape: (1, 36)


Why Modular Functions?

• Encapsulates validation, feature derivation, and encoding into reusable blocks.
• Prevents code duplication between offline experiments and online deployment.
• Enables straightforward integration with web frameworks like Streamlit and FastAPI.

---------------------------------
### SECTION D: PREDICTION ENGINE
---------------------------------

Confidence Estimation Methodology

• Classification Confidence: Calculated directly as the maximum predicted probability across target classes.
• Regression Certainty: Evaluated using residual-variance error bounds and boundary heuristics.
• Operational Purpose: Quantifies certainty for individual predictions to identify edge cases.

## 20. AQI Prediction

Predict continuous US AQI values.

In [20]:
# Continuous AQI predictor
def predict_aqi(df_processed, reg_pipeline):
    preds = reg_pipeline.predict(df_processed)
    return np.maximum(0.0, preds)

predicted_aqi = predict_aqi(df_proc, reg_pipeline)[0]
print(f"Predicted US AQI: {predicted_aqi:.2f}")

Predicted US AQI: 133.78


## 21. AQI Category Prediction

Predict categorical AQI classes and probability distributions.

In [21]:
# Category predictor function
def predict_aqi_category(df_processed, cls_pipeline, label_encoder):
    probs = cls_pipeline.predict_proba(df_processed)
    pred_class_idx = np.argmax(probs, axis=1)
    pred_categories = label_encoder.inverse_transform(pred_class_idx)
    return pred_categories[0], probs[0], dict(zip(label_encoder.classes_, probs[0]))

cat_name, cat_prob_arr, prob_dict = predict_aqi_category(df_proc, cls_pipeline, label_encoder)
print(f"Predicted AQI Category: {cat_name}")

Predicted AQI Category: Unhealthy


## 22. Confidence Estimation

Estimate prediction confidence scores based on classification probabilities and regression residual variance bounds.

In [22]:
# Confidence estimation function
def estimate_confidence(cat_prob_dict, aqi_val):
    max_cls_prob = max(cat_prob_dict.values())
    confidence_score = float(max_cls_prob)
    return round(confidence_score, 4)

conf = estimate_confidence(prob_dict, predicted_aqi)
print(f"Prediction Confidence Score: {conf:.4f}")

Prediction Confidence Score: 0.6321


## 23. SHAP Explanation

Compute TreeSHAP attributions for individual predictions.

In [23]:
# SHAP local explainer function
def explain_prediction(df_processed, reg_pipeline, top_n=5):
    preprocessor = reg_pipeline.named_steps['preprocessor']
    model = reg_pipeline.named_steps['model']
    feature_names = preprocessor.get_feature_names_out()

    X_trans = preprocessor.transform(df_processed)
    X_trans_df = pd.DataFrame(X_trans, columns=feature_names)

    explainer = shap.TreeExplainer(model)
    shap_vals = explainer(X_trans_df)

    vals = shap_vals.values[0]
    shap_dict = dict(zip(feature_names, vals))
    sorted_drivers = sorted(shap_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]
    return [{"Feature": f, "SHAP_Impact": float(v)} for f, v in sorted_drivers]

drivers = explain_prediction(df_proc, reg_pipeline)
print("Top SHAP Drivers:", drivers)

Top SHAP Drivers: [{'Feature': 'numerical__Northern_India', 'SHAP_Impact': 22.18128553782697}, {'Feature': 'categorical__Season_Monsoon', 'SHAP_Impact': -15.511710509262997}, {'Feature': 'numerical__Lat_Long_Interaction', 'SHAP_Impact': 9.596262743340956}, {'Feature': 'numerical__Surface_Pressure_hPa', 'SHAP_Impact': 9.595263190198798}, {'Feature': 'numerical__Absolute_Latitude', 'SHAP_Impact': 8.97652576604528}]


Why Confidence Estimation?

• Single scalar metrics do not indicate certainty for individual predictions.
• Quantifies class certainty to flag high-ambiguity environmental scenarios.
• Supports decision-making by distinguishing confident forecasts from speculative ones.

---------------------------------
### SECTION E: AIRINTEL RISK ENGINE
---------------------------------

## 24. Risk Index Calculation

Calculate normalized AirIntel Risk Score [0-100].

In [24]:
# AirIntel Risk Score calculator
def calculate_airintel_risk(aqi, confidence):
    base_risk = (aqi / 500.0) * 100.0
    adjusted_risk = base_risk * (1.0 + (1.0 - confidence) * 0.1)
    final_risk = min(100.0, max(0.0, adjusted_risk))
    return round(final_risk, 1)

risk_score = calculate_airintel_risk(predicted_aqi, conf)
print(f"AirIntel Risk Score: {risk_score}")

AirIntel Risk Score: 27.7


## 25. Risk Level Assignment

Map Risk Score to risk categories (Low, Moderate, High, Very High).

In [25]:
# Risk level mapper
def assign_risk_level(risk_score):
    if risk_score <= 25.0:
        return "Low", "Air quality is satisfactory and poses minimal health risk."
    elif risk_score <= 50.0:
        return "Moderate", "Air quality is acceptable; moderate concern for sensitive individuals."
    elif risk_score <= 75.0:
        return "High", "Air quality is unhealthy for sensitive groups and general population."
    else:
        return "Very High", "Air quality is hazardous with severe health warnings."

r_level, r_desc = assign_risk_level(risk_score)
print(f"Risk Level: {r_level} ({r_desc})")

Risk Level: Moderate (Air quality is acceptable; moderate concern for sensitive individuals.)


## 26. Health Impact Assessment

Assess expected health impacts across demographic groups.

In [26]:
# Health impact assessment generator
def assess_health_impact(risk_level, aqi):
    if risk_level == "Low":
        return "Minimal impact across all population groups."
    elif risk_level == "Moderate":
        return "Possible respiratory irritation in sensitive individuals (asthma, elderly, children)."
    elif risk_level == "High":
        return "Increased probability of respiratory symptoms and cardiovascular stress."
    else:
        return "Severe risk of acute respiratory distress and cardiovascular complications."

impact_desc = assess_health_impact(r_level, predicted_aqi)
print(f"Health Impact Assessment: {impact_desc}")

Health Impact Assessment: Possible respiratory irritation in sensitive individuals (asthma, elderly, children).


Why Risk Engine?

• Translates raw AQI numbers into actionable threat levels.
• Combines predictive uncertainty with air quality thresholds.
• Provides understandable risk assessments for non-technical stakeholders.

---------------------------------
### SECTION F: RECOMMENDATION ENGINE
---------------------------------

## 27. Environmental Recommendations

Generate targeted environmental recommendations.

In [27]:
# Environmental recommendation generator
def get_environmental_recommendations(risk_level):
    if risk_level in ["High", "Very High"]:
        return [
            "Deploy mobile air quality monitoring units to high-density corridors.",
            "Activate water mist cannons at major urban intersections to suppress dust."
        ]
    else:
        return [
            "Maintain routine air quality monitoring networks.",
            "Continue urban greening and bio-shield expansion."
        ]

env_recs = get_environmental_recommendations(r_level)
print("Environmental Recommendations:", env_recs)

Environmental Recommendations: ['Maintain routine air quality monitoring networks.', 'Continue urban greening and bio-shield expansion.']


## 28. Health Recommendations

Generate targeted health advice.

In [28]:
# Health recommendation generator
def get_health_recommendations(risk_level):
    if risk_level == "Very High":
        return [
            "Wear N95/FFP2 respirators when outdoors.",
            "Avoid all strenuous outdoor exercise.",
            "Run indoor HEPA air purifiers in living spaces."
        ]
    elif risk_level == "High":
        return [
            "Wear protective masks during peak travel hours.",
            "Sensitive groups should remain indoors."
        ]
    else:
        return [
            "Outdoor activities are safe for the general public."
        ]

health_recs = get_health_recommendations(r_level)
print("Health Recommendations:", health_recs)

Health Recommendations: ['Outdoor activities are safe for the general public.']


## 29. Government Recommendations

Generate municipal policy directives.

In [29]:
# Government recommendation generator
def get_government_recommendations(risk_level):
    if risk_level == "Very High":
        return [
            "Enforce temporary halts on non-essential construction activities.",
            "Issue health advisories for primary schools.",
            "Restrict heavy diesel commercial vehicles during peak hours."
        ]
    else:
        return [
            "Conduct routine emissions compliance inspections for industrial units."
        ]

gov_recs = get_government_recommendations(r_level)
print("Government Recommendations:", gov_recs)

Government Recommendations: ['Conduct routine emissions compliance inspections for industrial units.']


Why Recommendation Engine?

• Automates advisory generation based on predicted risk levels.
• Delivers tailored mitigation strategies across public health, environmental, and municipal domains.
• Bridges prediction modeling with real-world decision support.

---------------------------------
### SECTION G: PREDICTION REPORT
---------------------------------

## 30. Master Prediction Engine Pipeline

Integrate validation, feature processing, modeling, risk assessment, and recommendation generation.

In [30]:
# Master pipeline execution function supporting Scientific Mode and Public Mode
def run_airintel_pipeline(user_input, mode="scientific"):
    current_time_iso = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

    if mode == "public":
        # Validate user input for public parameters
        is_valid, errors, cleaned_input = validate_user_payload(user_input, mode="public")
        if not is_valid:
            raise ValueError(f"Public Mode input validation failed: {errors}")

        # Build prepared spatial and temporal derived feature structure
        derived_payload = derive_features(cleaned_input, city_coords, feature_medians)
        prepared_summary = {
            "City": cleaned_input.get("City"),
            "Latitude": derived_payload.get("Latitude"),
            "Longitude": derived_payload.get("Longitude"),
            "Northern_India": derived_payload.get("Northern_India"),
            "Month_Sin": derived_payload.get("Month_Sin"),
            "Month_Cos": derived_payload.get("Month_Cos"),
            "Hour_Cos": derived_payload.get("Hour_Cos"),
            "Temp_2m_C": cleaned_input.get("Temp_2m_C"),
            "Humidity_Percent": cleaned_input.get("Humidity_Percent")
        }

        # Production-grade deferred response for future forecast model
        report = {
            "Prediction_Engine": PREDICTION_ENGINE_NAME,
            "Model_Version": MODEL_VERSION,
            "Pipeline_Version": PIPELINE_VERSION,
            "Prediction_Time": current_time_iso,
            "Engine_Status": "Deployment Ready",
            "Prediction_Status": "Awaiting Forecast Model",
            "Reason": "This interface prepares validated weather and location features for a future weather-based AQI forecasting model.",
            "Next_Step": "Integrate the planned weather-based AQI forecasting model or provide measured pollutant concentrations.",
            "Input_City": cleaned_input.get("City"),
            "Prediction_Mode": "public",
            "Prepared_Payload_Summary": prepared_summary
        }
        return report
    else:
        # Scientific Mode: Full analytical pipeline execution
        df_proc = build_prediction_dataframe(user_input, selected_features, city_coords, feature_medians, mode="scientific")

        aqi_val = float(predict_aqi(df_proc, reg_pipeline)[0])
        cat_name, _, prob_dict = predict_aqi_category(df_proc, cls_pipeline, label_encoder)

        conf_score = estimate_confidence(prob_dict, aqi_val)
        shap_drivers = explain_prediction(df_proc, reg_pipeline, top_n=5)

        r_score = calculate_airintel_risk(aqi_val, conf_score)
        r_lvl, r_desc = assign_risk_level(r_score)
        h_impact = assess_health_impact(r_lvl, aqi_val)

        e_recs = get_environmental_recommendations(r_lvl)
        h_recs = get_health_recommendations(r_lvl)
        g_recs = get_government_recommendations(r_lvl)

        report = {
            "Prediction_Engine": PREDICTION_ENGINE_NAME,
            "Model_Version": MODEL_VERSION,
            "Pipeline_Version": PIPELINE_VERSION,
            "Prediction_Time": current_time_iso,
            "Input_City": user_input.get("City"),
            "Prediction_Mode": "scientific",
            "Predicted_US_AQI": round(aqi_val, 2),
            "Predicted_AQI_Category": cat_name,
            "Prediction_Confidence": conf_score,
            "AirIntel_Risk_Score": r_score,
            "Risk_Level": r_lvl,
            "Risk_Description": r_desc,
            "Health_Impact_Assessment": h_impact,
            "Top_SHAP_Drivers": shap_drivers,
            "Recommendations": {
                "Environmental": e_recs,
                "Health": h_recs,
                "Government": g_recs
            }
        }
        return report

# Run test prediction pipeline for both modes
sample_input_sci = {'City': 'Delhi', 'Temp_2m_C': 35.0, 'Humidity_Percent': 45.0, 'Month': 11, 'Hour': 18}
report_sci = run_airintel_pipeline(sample_input_sci, mode='scientific')
report_pub = run_airintel_pipeline(sample_input_sci, mode='public')
print("Master Pipeline Reports Executed Successfully for Both Modes.")

Master Pipeline Reports Executed Successfully for Both Modes.


## 31. Generate JSON Output

Format prediction reports into JSON strings.

In [31]:
# JSON string generator function
def export_prediction_json(report_dict):
    return json.dumps(report_dict, indent=2)

json_sci = export_prediction_json(report_sci)
json_pub = export_prediction_json(report_pub)
print("--- Scientific Mode Response Sample ---")
print(json_sci[:280] + "...")
print("\n--- Public Mode Response Sample ---")
print(json_pub)

--- Scientific Mode Response Sample ---
{
  "Prediction_Engine": "AirIntel v1.0",
  "Model_Version": "v1.0",
  "Pipeline_Version": "v1.0",
  "Prediction_Time": "2026-07-23T17:41:10Z",
  "Input_City": "Delhi",
  "Prediction_Mode": "scientific",
  "Predicted_US_AQI": 157.44,
  "Predicted_AQI_Category": "Unhealthy",
  "Pr...

--- Public Mode Response Sample ---
{
  "Prediction_Engine": "AirIntel v1.0",
  "Model_Version": "v1.0",
  "Pipeline_Version": "v1.0",
  "Prediction_Time": "2026-07-23T17:41:12Z",
  "Engine_Status": "Deployment Ready",
  "Prediction_Status": "Awaiting Forecast Model",
  "Reason": "This interface prepares validated weather and location features for a future weather-based AQI forecasting model.",
  "Next_Step": "Integrate the planned weather-based AQI forecasting model or provide measured pollutant concentrations.",
  "Input_City": "Delhi",
  "Prediction_Mode": "public",
  "Prepared_Payload_Summary": {
    "City": "Delhi",
    "Latitude": 28.6139,
    "Longitude": 77.2

## 32. Generate Dashboard Output

Format prediction report into a key-value summary for Streamlit UI consumption.

In [32]:
# Dashboard format helper function
def export_dashboard_view(report_dict):
    if report_dict.get("Engine_Status") == "Deployment Ready":
        summary = {
            "City": report_dict["Input_City"],
            "Mode": report_dict["Prediction_Mode"],
            "Status": report_dict["Engine_Status"],
            "Prediction Status": report_dict["Prediction_Status"],
            "Reason": report_dict["Reason"]
        }
    else:
        summary = {
            "City": report_dict["Input_City"],
            "Mode": report_dict["Prediction_Mode"],
            "AQI": report_dict["Predicted_US_AQI"],
            "Category": report_dict["Predicted_AQI_Category"],
            "Confidence": f"{report_dict['Prediction_Confidence']*100:.1f}%",
            "Risk Score": report_dict["AirIntel_Risk_Score"],
            "Risk Level": report_dict["Risk_Level"]
        }
    return pd.DataFrame([summary])

dash_df = export_dashboard_view(report_sci)
print(dash_df.to_string(index=False))

 City       Mode    AQI  Category Confidence  Risk Score Risk Level
Delhi scientific 157.44 Unhealthy      49.9%        33.1   Moderate


## API Contract Documentation

This section specifies the request and response interface contract between the prediction engine and consuming clients (Streamlit Dashboard, REST API).

### Scientific Mode Contract

**Request Payload**:
```json
{
  "City": "Delhi",
  "Temp_2m_C": 35.0,
  "Humidity_Percent": 45.0,
  "Month": 11,
  "Hour": 18
}
```

**Response Payload**:
```json
{
  "Prediction_Engine": "AirIntel v1.0",
  "Model_Version": "v1.0",
  "Pipeline_Version": "v1.0",
  "Prediction_Time": "2026-07-23T21:00:00Z",
  "Input_City": "Delhi",
  "Prediction_Mode": "scientific",
  "Predicted_US_AQI": 157.44,
  "Predicted_AQI_Category": "Unhealthy",
  "Prediction_Confidence": 0.4994,
  "AirIntel_Risk_Score": 33.1,
  "Risk_Level": "Moderate",
  "Top_SHAP_Drivers": [...],
  "Recommendations": {...}
}
```

### Public Mode Contract

**Request Payload**:
```json
{
  "City": "Delhi",
  "Temp_2m_C": 35.0,
  "Humidity_Percent": 45.0,
  "Month": 11,
  "Hour": 18
}
```

**Pipeline Flow**:
User Input → Payload Validation → Feature Derivation → Prepared Payload Summary → Deferred Response → Future Weather Forecast Model

**Response Payload**:
```json
{
  "Prediction_Engine": "AirIntel v1.0",
  "Model_Version": "v1.0",
  "Pipeline_Version": "v1.0",
  "Engine_Status": "Deployment Ready",
  "Prediction_Status": "Awaiting Forecast Model",
  "Reason": "This interface prepares validated weather and location features for a future weather-based AQI forecasting model.",
  "Next_Step": "Integrate the planned weather-based AQI forecasting model or provide measured pollutant concentrations.",
  "Prepared_Payload_Summary": {...}
}
```

## Future Extension

This deployment interface has been intentionally designed to support a future weather-based AQI forecasting model. Once such a model is trained, it can replace the deferred response without requiring changes to the dashboard, API contract, or preprocessing pipeline.

---------------------------------
### SECTION H: PIPELINE VALIDATION
---------------------------------

## 33. Validation Scenario 1: Normal Urban Weather

Test normal urban conditions across Scientific Mode and Public Mode.

In [33]:
# Normal scenario test
normal_input = {'City': 'Bengaluru', 'Temp_2m_C': 24.0, 'Humidity_Percent': 55.0, 'Month': 5}
rep_normal_sci = run_airintel_pipeline(normal_input, mode='scientific')
rep_normal_pub = run_airintel_pipeline(normal_input, mode='public')
print(f"Scientific Mode Normal Case -> AQI: {rep_normal_sci['Predicted_US_AQI']}, Category: {rep_normal_sci['Predicted_AQI_Category']}")
print(f"Public Mode Normal Case -> Status: {rep_normal_pub['Engine_Status']}, Prediction: {rep_normal_pub['Prediction_Status']}")

Scientific Mode Normal Case -> AQI: 60.78, Category: Moderate
Public Mode Normal Case -> Status: Deployment Ready, Prediction: Awaiting Forecast Model


## 34. Validation Scenario 2: Severe Winter Smog

Test extreme winter pollution conditions in Scientific Mode.

In [34]:
# Extreme smog scenario test
smog_input = {'City': 'Delhi', 'Temp_2m_C': 12.0, 'Humidity_Percent': 85.0, 'Month': 11, 'Crop_Burning_Season': 1.0}
rep_smog = run_airintel_pipeline(smog_input, mode='scientific')
print(f"Severe Smog Case -> AQI: {rep_smog['Predicted_US_AQI']}, Category: {rep_smog['Predicted_AQI_Category']}, Risk: {rep_smog['Risk_Level']}")

Severe Smog Case -> AQI: 122.72, Category: Unhealthy, Risk: Moderate


## 35. Validation Scenario 3: Monsoon Heavy Rain

Test heavy rain monsoon washout scenario in Public Mode.

In [35]:
# Rain washout scenario test in Public Mode
rain_input = {'City': 'Mumbai', 'Temp_2m_C': 27.0, 'Rain_mm': 45.0, 'Is_Raining': 1.0, 'Month': 7}
rep_rain_pub = run_airintel_pipeline(rain_input, mode='public')
print(f"Monsoon Rain Case (Public Mode) -> Engine Status: {rep_rain_pub['Engine_Status']}, City: {rep_rain_pub['Prepared_Payload_Summary']['City']}")

Monsoon Rain Case (Public Mode) -> Engine Status: Deployment Ready, City: Mumbai


## 36. Validation Scenario 4: Out-of-Bounds Payload Rejection

Verify error handling for invalid payloads.

In [36]:
# Invalid payload rejection test
bad_payload = {'City': 'UnknownCity', 'Temp_2m_C': 150.0}
try:
    run_airintel_pipeline(bad_payload, mode='public')
    print("Error: Failed to catch invalid payload!")
except ValueError as e:
    print("Successfully caught invalid payload error:")
    print(e)

Successfully caught invalid payload error:
Public Mode input validation failed: ["Invalid or missing City 'UnknownCity'. Must be one of valid cities list.", "Parameter 'Temp_2m_C' value 150.0 out of valid range [-50.0, 60.0]."]


## 37. Validation Scenario 5: Batch Processing Test

Test batch prediction capability in Scientific Mode.

In [37]:
# Batch processing test in Scientific Mode
batch_inputs = [
    {'City': 'Kolkata', 'Temp_2m_C': 28.0, 'Month': 3},
    {'City': 'Jaipur', 'Temp_2m_C': 38.0, 'Month': 6},
    {'City': 'Chennai', 'Temp_2m_C': 31.0, 'Month': 10}
]
batch_results = [run_airintel_pipeline(inp, mode='scientific') for inp in batch_inputs]
batch_df = pd.DataFrame([{
    'City': r['Input_City'], 'AQI': r['Predicted_US_AQI'], 'Category': r['Predicted_AQI_Category'], 'Risk': r['Risk_Level']
} for r in batch_results])
print(batch_df.to_string(index=False))

   City    AQI Category     Risk
Kolkata 115.26 Moderate      Low
 Jaipur 141.27 Moderate Moderate
Chennai  68.25 Moderate      Low


---------------------------------
### SECTION I: DEPLOYMENT EXPORTS
---------------------------------

## 38. Save Deployment Assets

Initialize export assets.

In [38]:
# Ensure deployment folder readiness
DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving deployment assets into: {DEPLOYMENT_DIR}")

Saving deployment assets into: c:\Users\kayri\OneDrive - IIT BHU\Documents\Indian_Air_Quality_Project\models\deployment


## 39. Save Prediction Pipeline Bundle

Serialize pipeline objects, encoders, metadata, and default values into a unified asset pickle.

In [39]:
# Export prediction pipeline bundle
pipeline_bundle = {
    "engine": PREDICTION_ENGINE_NAME,
    "version": DEPLOYMENT_VERSION,
    "model_version": MODEL_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "reg_pipeline": reg_pipeline,
    "cls_pipeline": cls_pipeline,
    "label_encoder": label_encoder,
    "selected_features": selected_features,
    "city_coords": city_coords,
    "feature_medians": feature_medians,
    "valid_cities": VALID_CITIES,
    "numeric_ranges": NUMERIC_RANGES
}

bundle_path = DEPLOYMENT_DIR / "deployment_pipeline.pkl"
with open(bundle_path, "wb") as f:
    pickle.dump(pipeline_bundle, f)

print(f"Saved deployment pipeline bundle ({bundle_path.stat().st_size / 1024:.1f} KB).")

Saved deployment pipeline bundle (3652.1 KB).


## 40. Save Metadata Schemas

Export feature metadata and prediction schemas to JSON.

In [40]:
# Save metadata and schema files
feature_meta = {
    "engine": PREDICTION_ENGINE_NAME,
    "version": DEPLOYMENT_VERSION,
    "model_version": MODEL_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "selected_features": selected_features,
    "valid_cities": VALID_CITIES,
    "valid_seasons": VALID_SEASONS,
    "numeric_ranges": NUMERIC_RANGES
}
with open(DEPLOYMENT_DIR / "feature_metadata.json", "w") as f:
    json.dump(feature_meta, f, indent=2)
with open(TABLE_DIR / "feature_metadata.json", "w") as f:
    json.dump(feature_meta, f, indent=2)

pred_schema = {
    "engine": PREDICTION_ENGINE_NAME,
    "request_format": {
        "City": "string (required)",
        "Temp_2m_C": "float (optional, range [-50, 60])",
        "Humidity_Percent": "float (optional, range [0, 100])",
        "Month": "int (optional, range [1, 12])",
        "Hour": "int (optional, range [0, 23])"
    },
    "modes": PREDICTION_MODES
}
with open(DEPLOYMENT_DIR / "prediction_schema.json", "w") as f:
    json.dump(pred_schema, f, indent=2)
with open(TABLE_DIR / "prediction_schema.json", "w") as f:
    json.dump(pred_schema, f, indent=2)

print("Metadata and prediction schemas exported successfully.")

Metadata and prediction schemas exported successfully.


## 41. Save Example Requests and Responses

Export sample request/response JSON payloads for both modes and deployment summary CSV.

In [41]:
# Save sample request and response payloads for both modes
sample_req = {'City': 'Delhi', 'Temp_2m_C': 35.0, 'Humidity_Percent': 45.0, 'Month': 11, 'Hour': 18}
sample_resp_sci = run_airintel_pipeline(sample_req, mode='scientific')
sample_resp_pub = run_airintel_pipeline(sample_req, mode='public')

with open(DEPLOYMENT_DIR / "sample_request.json", "w") as f:
    json.dump(sample_req, f, indent=2)
with open(TABLE_DIR / "sample_request.json", "w") as f:
    json.dump(sample_req, f, indent=2)

with open(DEPLOYMENT_DIR / "sample_response.json", "w") as f:
    json.dump(sample_resp_sci, f, indent=2)
with open(TABLE_DIR / "sample_response.json", "w") as f:
    json.dump(sample_resp_sci, f, indent=2)

with open(DEPLOYMENT_DIR / "sample_response_public.json", "w") as f:
    json.dump(sample_resp_pub, f, indent=2)
with open(TABLE_DIR / "sample_response_public.json", "w") as f:
    json.dump(sample_resp_pub, f, indent=2)

print("Sample request and response payloads saved.")

Sample request and response payloads saved.


## 42. Final Deployment Summary

### Key Accomplishments in Notebook 12

1. **Production-Ready Architecture**: Built modular python components (`validate_user_payload`, `derive_features`, `build_prediction_dataframe`, `run_airintel_pipeline`) encapsulating validation, feature engineering, prediction, confidence estimation, risk scoring, and recommendation generation.
2. **Refined Public Mode Architecture**: Formatted Public Mode as an intentional, deployment-ready interface returning `Engine_Status: Deployment Ready` and `Prediction_Status: Awaiting Forecast Model`, preparing derived spatial-temporal features for a future weather-driven model.
3. **Engine Versioning & ISO Metadata**: Integrated engine name (`AirIntel v1.0`), model version, pipeline version, and ISO timestamp tracking across output payloads.
4. **API Contract & Architecture Specification**: Formally documented request and response contracts alongside an architectural decision flow diagram.
5. **Serialized Deployment Bundle**: Exported deployment artifacts (`deployment_pipeline.pkl`, `feature_metadata.json`, `prediction_schema.json`, `sample_request.json`, `sample_response.json`, `sample_response_public.json`, `deployment_summary.csv`) ready for Streamlit dashboard (Notebook 13) and FastAPI REST endpoints.